# Supplementary Table 3

PREDICT antibiotic endpoints (human-assigned share of total LFQ intensity and quantified protein groups) under four definitions of antibiotic exposure that differ only in how rifaximin recorded for hepatic encephalopathy is handled. The first section rebuilds the published workbook from `source_data/STab3/` and checks it cell by cell. The second recomputes every statistic and count from the controlled PREDICT inputs when `FASTALAKE_CONTROLLED_DIR` is set, and otherwise skips.

Run from the repository root: `python paper.py build STab3`.

In [ ]:
from pathlib import Path
import sys
UNIT = Path.cwd()                 # the notebook runs with its own directory as cwd
REPO = UNIT.parents[1]
sys.path[:0] = [str(REPO), str(UNIT)]
SD = REPO / "source_data" / UNIT.name
OUT = REPO / "build" / UNIT.name   # rendered outputs; gitignored
OUT.mkdir(parents=True, exist_ok=True)
from style import fl_style as fl
import panels
fl.init()

## Table

Rebuilds `Supplementary_Table_3.xlsx` (sheets Read_me, Endpoints, Composition) from `endpoints_by_definition.tsv` and `composition_by_definition.tsv` with the code that assembled the published workbook, then compares every cell value with `expected/Supplementary_Table_3.xlsx`. Receipt: `checks/workbook.json`.

In [ ]:
import json
import pandas as pd
built = panels.build_workbook(SD, OUT / "Supplementary_Table_3.xlsx")
receipt = panels.compare_workbooks(built, UNIT / "expected" / "Supplementary_Table_3.xlsx")
receipt.update(built="build/STab3/Supplementary_Table_3.xlsx", expected="tables/STab3/expected/Supplementary_Table_3.xlsx")
(UNIT / "checks").mkdir(exist_ok=True)
(UNIT / "checks" / "workbook.json").write_text(json.dumps(receipt, indent=1) + "\n")
print(receipt["status"], receipt["cells_compared"], "cells,", receipt["n_differences"], "differences")
pd.set_option("display.width", 250, "display.max_columns", 30)
panels.read_endpoints(SD)[["scenario", "metric", "n_patients", "mean_on_minus_off", "CI95_low", "CI95_high",
                          "p_signflip_MC", "q_BH_two_endpoints_within_definition"]]

## Statistics (controlled inputs)

Supports Results, v28d line 137: "Counting rifaximin given for hepatic encephalopathy as antibiotic exposure left 40 paired patients, with an 11.5 percentage-point higher human-assigned share (95% CI 5.7–17.7, q=0.00048) and 2,378 fewer protein groups (95% CI 1,140–3,651, q=0.00054, Supplementary Table 3)", and the counts in Methods, v28d line 793. Unit: patient; n = 42, 40, 35 and 38 paired patients (definitions A to D) from 306 acquisitions. Test: mean paired on-minus-off difference, percentile patient bootstrap (50,000 draws), two-sided Monte Carlo sign-flip test (200,000 draws, plus-one correction), Benjamini–Hochberg across the two endpoints within each definition. A planted negative control (every hepatic-encephalopathy flag inverted) must leave definition A unchanged and change definition B.

Needs `PREDICT_metadata_2026-08-03.tsv` and `acquisition_accounting.tsv` in `FASTALAKE_CONTROLLED_DIR` (see `controlled/CONTROLLED_INPUTS.md`). Only aggregate values are printed.

In [ ]:
primary = pd.read_csv(SD / "primary_paired_statistics.tsv", sep="\t")
loaded = panels.load_controlled()
res = None
if loaded is not None:
    d, ac = loaded
    res = panels.stat_definitions(d, ac, primary)
    cmp = panels.compare_to_source(res, SD)
    print(json.dumps(cmp, indent=1))
    assert cmp["stats_identical_to_producer_output"] and cmp["composition_identical"] and cmp["counts_identical"]
    neg = panels.stat_definitions(d, ac, primary, negative_control=True)
    print("negative control: A n =", neg["counts"]["paired_A_current"]["n"], "| B n =", neg["counts"]["paired_B_HE_as_exposure"]["n"])
    assert neg["counts"]["paired_A_current"]["n"] == 42 and neg["counts"]["paired_B_HE_as_exposure"]["n"] != 40
    del d, ac
    display(res["stats"][["scenario", "metric", "n_patients", "mean_on_minus_off", "CI95_low", "CI95_high",
                          "signflip_extreme_count", "p_signflip_MC", "q_two_clinical_endpoints", "acq_off", "acq_on"]])

## Checks and numbers

The table check runs above (`checks/workbook.json`); this unit has no figure panels, so there are no pixel checks. The numbers below are printed in `NUMBERS.tsv` order, at full precision, from the controlled recomputation.

In [ ]:
print("workbook check:", receipt["status"])
if res is None:
    print("numbers SKIPPED: controlled inputs absent")
else:
    for label, value in panels.numbers(res):
        print(f"{label}\t{value!r}")